# F.A.B.L.E. — Federated Agent Bus & Lifecycle Engine

**Multi-agent LLM orchestration** via OpenRouter | Claude + Llama 3 + GPT-4o-mini | RAG | Obsidian UI

This notebook demonstrates the core F.A.B.L.E. pipeline:
1. **Analyst** — structured analysis with retrieved context (Claude Sonnet)
2. **Critic** — targeted critique of the analysis (Llama 3 70B)
3. **Synthesizer** — final actionable response (Claude Sonnet)

All models routed through [OpenRouter](https://openrouter.ai) — one API key, many models.

In [ ]:
!pip install openai sentence-transformers faiss-cpu structlog pydantic-settings -q

In [ ]:
import os
# Set your OpenRouter API key (get one at https://openrouter.ai/keys)
os.environ['OPENROUTER_API_KEY'] = 'YOUR_KEY_HERE'

In [ ]:
# Clone the repo (or upload the backend/ directory)
# !git clone https://github.com/YOUR_USERNAME/FABLE.git
# import sys; sys.path.insert(0, 'FABLE')

# For Kaggle: assume backend/ is available in the working directory
import sys
sys.path.insert(0, '.')

In [ ]:
import asyncio
from backend.agents.register import register_all
from backend.core.lifecycle import run_task

register_all()
print('Agents registered: analyst, critic, synthesizer')

## Domain 1 — Code Review


In [ ]:
CODE_SAMPLE = '''
def get_user(username):
    query = f"SELECT * FROM users WHERE username = '{username}'"
    return db.execute(query)
'''

result = asyncio.run(run_task(CODE_SAMPLE, domain='code_review'))

for msg in result['messages']:
    print(f"\n{'='*60}")
    print(f"[{msg['role'].upper()}]")
    print('='*60)
    print(msg['content'])

## Domain 2 — Finance


In [ ]:
FINANCE_QUERY = 'Analyze NVIDIA revenue growth sustainability given the 2024 data center boom.'

result = asyncio.run(run_task(FINANCE_QUERY, domain='finance'))

for msg in result['messages']:
    print(f"\n{'='*60}")
    print(f"[{msg['role'].upper()}]")
    print('='*60)
    print(msg['content'])

## Evaluation — Multi-agent vs Single-agent


In [ ]:
from backend.evaluation.rubric import score

task = CODE_SAMPLE
multi_output = result['messages'][-1]['content']

# Score the synthesizer output
scores = asyncio.run(score(task, multi_output))
print('Multi-agent scores:')
for k, v in scores.items():
    bar = '█' * int(v * 20)
    print(f'  {k:15s} {bar:<20s} {v:.2f}')

## Export this run as a notebook

Use `export_notebook.py` to convert any logged run to a shareable `.ipynb`.


In [ ]:
from backend.evaluation.export_notebook import export_all
paths = export_all(output_dir='./output_notebooks')
print(f'Exported {len(paths)} notebook(s)')